In [ ]:
pip install jira

In [ ]:
from jira import JIRA

# --- Jira Cloud connection ---

# Pull secrets from Databricks

jira_url = <jira URL>
email = <email address>
token = dbutils.secrets.get(scope = "anna.lambert", key = "jira-token")

jira = JIRA(server=jira_url, basic_auth=(email, token))

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW Weekly_cancelled_shipments AS
 SELECT S.shipment_id
      ,S.ref_id AS shipment_reference
      ,u.email AS cancelled_by_email
      ,u.first_name AS cancelled_by_first_name
      ,u.last_name AS cancelled_by_last_name
      , FROM_UNIXTIME(L.cts) AS cancellation_datetime
      ,L.datastr
      ,CASE WHEN S.excursion_severity = 0 THEN "No Excursion"
						WHEN S.excursion_severity = 1 THEN "Low Excursion"
						WHEN S.excursion_severity = 2 THEN "Medium Excursion"
						WHEN S.excursion_severity = 3 THEN "High Excursion"
						END AS Excursion_Type
       ,S.from_location_id AS Origin_Location_ID
       ,S.from_name AS Origin_Name
       ,S.from_country AS Origin_Country
       ,S.to_location_id AS Destination_Location_ID
       ,S.to_name AS Destination_Name
       ,S.to_country AS Destination_Country
      ,CASE WHEN S.quality = 1 THEN "OK"
                        WHEN S.quality = 2 THEN "Unknown"
                        WHEN S.quality = 3 THEN "Bad"
                        WHEN S.quality = 4 THEN "Undecided"
                        WHEN S.quality = 5 THEN "Released"
                        WHEN S.quality = 6 THEN "Rejected"
                        WHEN S.quality = 7 THEN "Partly Released"
                        END AS Shipment_Quality
      , CASE WHEN S.status = 2 THEN "Shipping"
							WHEN S.status = 3 THEN "Delivered"
							WHEN S.status = 4 THEN "Closed"
							WHEN S.status = 5 THEN "Deleted"
							WHEN S.status = 6 THEN "Draft"
							WHEN S.status = 7 THEN "Ready"
							END AS Shipment_Status
      --,S.create_datetime
     -- ,S.ready_datetime
      ,FROM_UNIXTIME(S.shipped_ts) AS shipped_datetime
      ,FROM_UNIXTIME(S.delivered_ts) AS delivered_datetime
      ,FROM_UNIXTIME(S.closed_ts) AS closed_datetime
  FROM mysql.silver.shipments S
  --LEFT JOIN pbi.v_AuditLog L on s.shipment_id = l.resource_id AND l.resource_type = 'SHIPMENT' AND (l.description LIKE '%to Deleted%' OR l.description LIKE '%Shipment deleted%') -- cancelled by users or integration
  LEFT JOIN mysql.silver.live_auditlog L on s.shipment_id = l.data_id AND l.data_type = 'Shipment' AND  l.datastr LIKE '%Shipment deleted%' -- cancelled by integration
  INNER JOIN mysql.silver.live_users U ON U.user_id=L.user_id
  WHERE  S.customer_id = 123 AND s.status IN (5)
  AND FROM_UNIXTIME(L.cts) BETWEEN  date_sub(current_date(), 7) AND date_sub(current_date(), 1)

In [ ]:
%python

df1 = spark.sql("SELECT * FROM Weekly_cancelled_shipments")

In [ ]:
%pip install xlsxwriter

In [ ]:
import pandas as pd
import tempfile
from datetime import datetime
import shutil
import os

# Convert Spark DataFrames to Pandas DataFrames
pdf1 = df1.toPandas()

# Generate dated filename
current_date = datetime.now().strftime("%Y%m%d")
file_name = f"Weekly_cancelled_shipments_by_integration_{current_date}.xlsx"
output_path = f"/tmp/{file_name}"

# Create a temporary directory
with tempfile.TemporaryDirectory() as tmpdirname:
    # Define the output path in the temporary directory
    output_path = "/tmp/"+file_name


    # Save to Excel with multiple sheets
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        pdf1.to_excel(writer, index=False)


In [ ]:
# Jira issue key
issue_key = 'JIRA-1234'


# Attach the file
jira.add_attachment(issue=issue_key, attachment=output_path)

In [ ]:
# Get the issue details
issue = jira.issue(issue_key)

# Find the attachment
for attachment in issue.fields.attachment:
    if attachment.filename == f"Weekly_cancelled_shipments_by_integration_{current_date}.xlsx":
        csv_url = attachment.content
        break


# Add a comment with the link
comment_text = f"An updated file has been attached:  [data.csv|{csv_url}]. If the file does not list any shipments, then no shipment has been cancelled by integration in the previous week."
jira.add_comment(issue_key, comment_text)

print(f"✅ Comment under {issue_key} posted")